# Resume Parsing

**Goal:** Extract structured information (Name, Email, Phone, Skills, Education, Experience, Projects, Certifications) from PDF and DOCX resumes into a clean JSON format.

## Why this module exists
Up to now, we used Kaggle CSV data containing raw text. However, a real-world ATS (Applicant Tracking System) receives resumes as PDFs or Word documents. To make the AI Hiring Assistant usable, we need a robust parsing engine to read these files and structure the unstructured text.

## How it integrates
The extracted structured JSON is the input for **# (Data Preprocessing)** and subsequently **# (Classification)** and **# (Ranking)**. The raw extracted text also feeds into **# (RAG Pipeline)** for providing contextual answers to candidates.

## Algorithms used
- **PyMuPDF (`fitz`) / `pdfplumber`**: For high-fidelity PDF text and layout extraction.
- **`python-docx`**: For extracting text from Microsoft Word documents.
- **spaCy (NLP)**: Named Entity Recognition (NER) to find names and organizations.
- **Regex Pattern Matching**: To reliably extract emails, phone numbers, and URLs.
- **Rule-based Section Heuristics**: To segment the resume into Education, Experience, Projects, etc.

## Inputs & Outputs
- **Input**: `.pdf` or `.docx` resume files.
- **Output**: Structured JSON containing extracted fields.


In [1]:
import os
import json
import re
import spacy
import fitz  # PyMuPDF
import pdfplumber
import docx
from docx import Document

# Ensure directories exist
os.makedirs('../reports/Module_6', exist_ok=True)
os.makedirs('../data/sample_resumes', exist_ok=True)

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Downloading spaCy model...")
    import spacy.cli
    spacy.cli.download("en_core_web_sm")
    nlp = spacy.load("en_core_web_sm")


## 1. Create Dummy Resumes for Testing

In [2]:
# Let's create a dummy DOCX resume
doc = Document()
doc.add_heading('John Doe', 0)
doc.add_paragraph('Email: john.doe@example.com | Phone: (123) 456-7890 | LinkedIn: linkedin.com/in/johndoe')
doc.add_heading('Skills', level=1)
doc.add_paragraph('Python, Machine Learning, Data Science, AWS, SQL')
doc.add_heading('Experience', level=1)
doc.add_paragraph('Data Scientist at Tech Corp (2020 - Present)')
doc.add_paragraph('- Developed ML models for customer churn.\n- Built RAG pipelines using LangChain.')
doc.add_heading('Education', level=1)
doc.add_paragraph('M.S. in Computer Science, University of Technology (2018 - 2020)')
doc.save('../data/sample_resumes/John_Doe_Resume.docx')

# We'll use reportlab to create a dummy PDF
from reportlab.pdfgen import canvas

c = canvas.Canvas('../data/sample_resumes/Jane_Smith_Resume.pdf')
c.drawString(100, 800, "Jane Smith")
c.drawString(100, 780, "Email: jane.smith@demo.com | Phone: +1-987-654-3210")
c.drawString(100, 750, "SKILLS")
c.drawString(100, 735, "Java, Spring Boot, Microservices, Docker, Kubernetes")
c.drawString(100, 705, "EXPERIENCE")
c.drawString(100, 690, "Senior Backend Engineer at Dev Solutions (2019 - Present)")
c.drawString(100, 675, "- Designed scalable REST APIs.")
c.drawString(100, 645, "EDUCATION")
c.drawString(100, 630, "B.S. in Software Engineering, State University (2015 - 2019)")
c.save()
print("Sample PDF and DOCX generated.")


Sample PDF and DOCX generated.


## 2. Text Extraction Functions

In [3]:
def extract_text_from_pdf(pdf_path):
    text = ""
    # Try PyMuPDF first
    try:
        doc = fitz.open(pdf_path)
        for page in doc:
            text += page.get_text() + "\n"
    except Exception as e:
        # Fallback to pdfplumber
        try:
            with pdfplumber.open(pdf_path) as pdf:
                for page in pdf.pages:
                    text += page.extract_text() + "\n"
        except Exception as e2:
            print(f"Error extracting PDF: {e2}")
    return text.strip()

def extract_text_from_docx(docx_path):
    text = ""
    try:
        doc = Document(docx_path)
        for para in doc.paragraphs:
            text += para.text + "\n"
    except Exception as e:
        print(f"Error extracting DOCX: {e}")
    return text.strip()

def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    if ext == '.pdf':
        return extract_text_from_pdf(file_path)
    elif ext == '.docx':
        return extract_text_from_docx(file_path)
    else:
        return ""


## 3. Information Extraction Logic

In [4]:
def extract_email(text):
    match = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', text)
    return match.group(0) if match else None

def extract_phone(text):
    match = re.search(r'\+?\d{1,3}[-.\s]?\(?\d{1,4}?\)?[-.\s]?\d{1,4}[-.\s]?\d{1,9}', text)
    return match.group(0) if match else None

def extract_sections(text):
    # A simple rule-based segmenter
    lines = text.split('\n')
    sections = {
        'Skills': [],
        'Experience': [],
        'Education': [],
        'Projects': [],
        'Certifications': []
    }
    
    current_section = None
    section_keywords = {
        'Skills': ['skills', 'technologies', 'core competencies'],
        'Experience': ['experience', 'work history', 'employment'],
        'Education': ['education', 'academic background'],
        'Projects': ['projects', 'personal projects'],
        'Certifications': ['certifications', 'licenses', 'courses']
    }
    
    for line in lines:
        line_clean = line.strip().lower()
        if not line_clean:
            continue
            
        found_header = False
        for sec, keywords in section_keywords.items():
            if any(line_clean.startswith(kw) for kw in keywords) and len(line_clean.split()) < 5:
                current_section = sec
                found_header = True
                break
                
        if not found_header and current_section:
            sections[current_section].append(line.strip())
            
    # Join lists back into strings
    for sec in sections:
        sections[sec] = '\n'.join(sections[sec]).strip()
        
    return sections

def parse_resume(file_path):
    raw_text = extract_text(file_path)
    if not raw_text:
        return {}
        
    # spaCy for NLP Name Extraction (Heuristically taking top PERSON entity)
    doc = nlp(raw_text)
    name = None
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            name = ent.text
            break
            
    sections = extract_sections(raw_text)
    
    return {
        "filename": os.path.basename(file_path),
        "name": name,
        "email": extract_email(raw_text),
        "phone": extract_phone(raw_text),
        "skills": sections.get("Skills", ""),
        "experience": sections.get("Experience", ""),
        "education": sections.get("Education", ""),
        "projects": sections.get("Projects", ""),
        "certifications": sections.get("Certifications", ""),
        "raw_text": raw_text
    }


## 4. Test Parser on Sample Resumes

In [5]:
files = ['../data/sample_resumes/John_Doe_Resume.docx', '../data/sample_resumes/Jane_Smith_Resume.pdf']
results = []

for f in files:
    res = parse_resume(f)
    # Remove raw text for cleaner JSON output preview
    preview = res.copy()
    preview.pop('raw_text', None)
    results.append(preview)
    
print(json.dumps(results, indent=2))

# Save output
with open('../reports/Module_6/sample_extracted_resumes.json', 'w') as out_f:
    json.dump(results, out_f, indent=2)


[
  {
    "filename": "John_Doe_Resume.docx",
    "name": "John Doe\nEmail",
    "email": "john.doe@example.com",
    "phone": "123) 456-7890",
    "skills": "Python, Machine Learning, Data Science, AWS, SQL",
    "experience": "Data Scientist at Tech Corp (2020 - Present)\n- Developed ML models for customer churn.\n- Built RAG pipelines using LangChain.",
    "education": "M.S. in Computer Science, University of Technology (2018 - 2020)",
    "projects": "",
    "certifications": ""
  },
  {
    "filename": "Jane_Smith_Resume.pdf",
    "name": "Jane Smith",
    "email": "jane.smith@demo.com",
    "phone": "+1-987-654",
    "skills": "Java, Spring Boot, Microservices, Docker, Kubernetes",
    "experience": "Senior Backend Engineer at Dev Solutions (2019 - Present)\n- Designed scalable REST APIs.",
    "education": "B.S. in Software Engineering, State University (2015 - 2019)",
    "projects": "",
    "certifications": ""
  }
]


## 5. Save as Python Module

In [6]:
# Write the core parsing logic to src/resume_parser.py
parser_code = """import os
import re
import spacy
import fitz  # PyMuPDF
import pdfplumber
from docx import Document

nlp = spacy.load("en_core_web_sm")

def extract_text(file_path):
    ext = os.path.splitext(file_path)[1].lower()
    text = ""
    if ext == '.pdf':
        try:
            doc = fitz.open(file_path)
            for page in doc:
                text += page.get_text() + "\\n"
        except Exception:
            try:
                with pdfplumber.open(file_path) as pdf:
                    for page in pdf.pages:
                        text += page.extract_text() + "\\n"
            except Exception:
                pass
    elif ext == '.docx':
        try:
            doc = Document(file_path)
            for para in doc.paragraphs:
                text += para.text + "\\n"
        except Exception:
            pass
    return text.strip()

def extract_email(text):
    match = re.search(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\\.[a-zA-Z]{2,}', text)
    return match.group(0) if match else None

def extract_phone(text):
    match = re.search(r'\\+?\\d{1,3}[-.\\s]?\\(?\\d{1,4}?\\)?[-.\\s]?\\d{1,4}[-.\\s]?\\d{1,9}', text)
    return match.group(0) if match else None

def extract_sections(text):
    lines = text.split('\\n')
    sections = {'Skills': [], 'Experience': [], 'Education': [], 'Projects': [], 'Certifications': []}
    current_section = None
    section_keywords = {
        'Skills': ['skills', 'technologies'],
        'Experience': ['experience', 'work history'],
        'Education': ['education'],
        'Projects': ['projects'],
        'Certifications': ['certifications']
    }
    for line in lines:
        line_clean = line.strip().lower()
        if not line_clean: continue
        found_header = False
        for sec, keywords in section_keywords.items():
            if any(line_clean.startswith(kw) for kw in keywords) and len(line_clean.split()) < 5:
                current_section = sec
                found_header = True
                break
        if not found_header and current_section:
            sections[current_section].append(line.strip())
            
    for sec in sections:
        sections[sec] = '\\n'.join(sections[sec]).strip()
    return sections

def parse_resume(file_path):
    raw_text = extract_text(file_path)
    if not raw_text: return {}
    doc = nlp(raw_text)
    name = None
    for ent in doc.ents:
        if ent.label_ == "PERSON":
            name = ent.text
            break
    sections = extract_sections(raw_text)
    return {
        "filename": os.path.basename(file_path),
        "name": name,
        "email": extract_email(raw_text),
        "phone": extract_phone(raw_text),
        "skills": sections.get("Skills", ""),
        "experience": sections.get("Experience", ""),
        "education": sections.get("Education", ""),
        "projects": sections.get("Projects", ""),
        "certifications": sections.get("Certifications", "")
    }
"""
with open('../src/resume_parser.py', 'w', encoding='utf-8') as f:
    f.write(parser_code)
print("Saved parser module to src/resume_parser.py")


Saved parser module to src/resume_parser.py


# Candidate Ranking

**Goal:** Create a weighted ranking algorithm to rank candidates for specific Job Descriptions based on Semantic Similarity, Skill Match, Experience, and Education.

## Why this module exists
A single similarity score is often not enough to make hiring decisions. Recruiters care about hard skills, years of experience, and education levels. By combining semantic matching with explicit feature matching, we create a more robust and explainable ranking system.

## How it integrates
It consumes the Semantic Similarity embeddings/scores from# and the extracted features (skills, experience keywords, education keywords) from# to produce the final, holistic ranking score that will power the Streamlit demo (# ).

## Algorithms used
- **Weighted Scoring Model**: Final Score = w1 * Semantic Similarity + w2 * Skill Match + w3 * Experience Match + w4 * Education Match
- **Jaccard Similarity / Overlap**: For Skill Match (intersection over required skills).
- **Min-Max Scaling**: To normalize experience and education counts into 0-1 ranges.

## Inputs & Outputs
- **Input**: `train.csv` (features), `similarity report.csv` or newly computed similarities.
- **Output**: Ranked Top Candidates Report (`.csv`), Ranking Visualizations (`.png`).


In [1]:
import pandas as pd
import numpy as np
import os
import ast
import joblib
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Ensure directories exist
os.makedirs('../reports/Module_5', exist_ok=True)


## 1. Define Job Descriptions with Requirements

In [2]:
job_descriptions = [
    {
        "jd_id": "JD_1",
        "title": "Data Scientist",
        "text": "Looking for a Data Scientist with experience in Machine Learning, Python, Pandas, Scikit-Learn, and Deep Learning.",
        "required_skills": ["python", "machine learning", "pandas", "scikit-learn", "deep learning", "sql"],
        "ideal_experience_keywords": 4,
        "ideal_education_keywords": 2
    },
    {
        "jd_id": "JD_2",
        "title": "Backend Developer",
        "text": "Need a Backend Developer skilled in Node.js, Express, MongoDB, and RESTful APIs.",
        "required_skills": ["node.js", "express", "mongodb", "api", "rest", "javascript", "backend"],
        "ideal_experience_keywords": 3,
        "ideal_education_keywords": 1
    },
    {
        "jd_id": "JD_3",
        "title": "Cloud Engineer",
        "text": "Hiring a Cloud Engineer with AWS, Docker, Kubernetes, and Terraform experience.",
        "required_skills": ["aws", "docker", "kubernetes", "terraform", "cloud", "linux", "ci/cd"],
        "ideal_experience_keywords": 5,
        "ideal_education_keywords": 1
    }
]


## 2. Load Data and Embeddings

In [3]:
print("Loading data...")
try:
    df = pd.read_csv('../data/processed/train.csv').fillna("")
except:
    df = pd.read_csv('../data/processed/resumes_eda_pass.csv').fillna("")

# Load embeddings
embeddings = joblib.load('../vector_store/st_resume_embeddings.pkl')
model = SentenceTransformer('all-MiniLM-L6-v2')


Loading data...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

## 3. Calculate Matches and Scores

In [4]:
# Define weights
W_SEMANTIC = 0.40
W_SKILL = 0.35
W_EXPERIENCE = 0.15
W_EDUCATION = 0.10

def calculate_skill_match(resume_skills_str, required_skills):
    if not isinstance(resume_skills_str, str) or not resume_skills_str:
        return 0.0
    
    resume_skills = [s.strip().lower() for s in resume_skills_str.split(',')]
    req_skills = [s.lower() for s in required_skills]
    
    # How many of the required skills are in the resume?
    matched = sum(1 for req in req_skills if any(req in res or res in req for res in resume_skills))
    
    return min(1.0, matched / len(req_skills))

all_ranked_results = []

for jd in job_descriptions:
    # 1. Semantic Similarity
    jd_emb = model.encode([jd["text"]])
    sim_scores = cosine_similarity(jd_emb, embeddings).flatten()
    
    jd_df = df[['ResumeID', 'Category', 'extracted_skills', 'experience_keyword_count', 'education_keyword_count']].copy()
    jd_df['Semantic_Score'] = sim_scores
    
    # 2. Skill Match
    jd_df['Skill_Match_Score'] = jd_df['extracted_skills'].apply(lambda x: calculate_skill_match(x, jd['required_skills']))
    
    # 3. Experience Match (Normalize based on ideal, cap at 1.0)
    jd_df['Experience_Score'] = (jd_df['experience_keyword_count'] / jd['ideal_experience_keywords']).clip(upper=1.0)
    
    # 4. Education Match (Normalize based on ideal, cap at 1.0)
    jd_df['Education_Score'] = (jd_df['education_keyword_count'] / jd['ideal_education_keywords']).clip(upper=1.0)
    
    # 5. Final Weighted Score
    jd_df['Final_Score'] = (
        (W_SEMANTIC * jd_df['Semantic_Score']) +
        (W_SKILL * jd_df['Skill_Match_Score']) +
        (W_EXPERIENCE * jd_df['Experience_Score']) +
        (W_EDUCATION * jd_df['Education_Score'])
    )
    
    jd_df['JD_ID'] = jd['jd_id']
    jd_df['JD_Title'] = jd['title']
    
    # Sort and pick top candidates
    top_candidates = jd_df.sort_values(by='Final_Score', ascending=False).head(10)
    all_ranked_results.append(top_candidates)

final_ranking_df = pd.concat(all_ranked_results, ignore_index=True)
final_ranking_df.to_csv('../reports/Module_5/top_candidates_ranking.csv', index=False)
print("Ranking completed and saved.")


Ranking completed and saved.


## 4. Visualizations

In [5]:
# Plot ranking components for the top 5 candidates of JD_1
top_jd1 = final_ranking_df[final_ranking_df['JD_ID'] == 'JD_1'].head(5)

# Melt the dataframe for seaborn plotting
plot_df = top_jd1[['ResumeID', 'Semantic_Score', 'Skill_Match_Score', 'Experience_Score', 'Education_Score']].copy()
# Apply weights for visualization to show contribution
plot_df['Semantic (40%)'] = plot_df['Semantic_Score'] * W_SEMANTIC
plot_df['Skills (35%)'] = plot_df['Skill_Match_Score'] * W_SKILL
plot_df['Experience (15%)'] = plot_df['Experience_Score'] * W_EXPERIENCE
plot_df['Education (10%)'] = plot_df['Education_Score'] * W_EDUCATION

plot_df = plot_df.drop(columns=['Semantic_Score', 'Skill_Match_Score', 'Experience_Score', 'Education_Score'])
plot_df.set_index('ResumeID').plot(kind='bar', stacked=True, figsize=(10, 6), colormap='viridis')

plt.title('Score Breakdown for Top 5 Data Scientist Candidates')
plt.ylabel('Final Weighted Score')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('../reports/Module_5/score_breakdown_jd1.png')
plt.show()


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_21080\4187420663.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
